# Run your own scenarios through the EWR dashboard

Runs py_ewr on your model output file(s) and writes a copy of the dashboard with the results built in, flow charts included. No server or command line is needed, and the saved file opens by double-click.

1. Upload your file(s) (a Unity Catalog volume, or the workspace). Note the path, e.g. `/Volumes/<catalog>/<schema>/<volume>/run.csv`.
2. Edit `SCENARIOS` in the `SCENARIOS` cell, then **Run all**.
3. The page is shown below and saved as `ewr_dashboard_with_scenarios.html`; download that file to open it in your browser.

In [ ]:
%pip install py_ewr pandas numpy

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Edit this list. "path" is the full path of one file: /Volumes/..., /Workspace/..., /dbfs/... or dbfs:/... "format" is one of: Standard time-series, Bigmod - MDBA, FIRM - MDBA, res.csv - MDBA,
# Source - NSW (res.csv), IQQM - netcdf, ten thousand year.  "name" is optional (defaults to the file name).
SCENARIOS = [
    {"name": "My scenario", "path": "/Volumes/<catalog>/<schema>/<volume>/my_run.csv", "format": "Standard time-series"},
    # {"name": "Wetter option", "path": "/Volumes/<catalog>/<schema>/<volume>/wetter.csv", "format": "Standard time-series"},
]
# Where the page is written. The default is the repository folder this notebook is in; use a /Volumes/... path to keep it elsewhere.
OUT_FILE = None

In [ ]:
import os, sys
def find_repo():   # the repository folder: from the notebook's own path when Databricks gives it, else the working folder or the folders above it
    starts = []
    try:
        starts.append("/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()))
    except Exception:
        pass
    starts.append(os.getcwd())
    for start in starts:
        d = start
        for _ in range(4):
            if os.path.isfile(os.path.join(d, "src", "embed_scenarios.py")):
                return d
            d = os.path.dirname(d)
    raise FileNotFoundError("Could not find the repository folder (the one containing src/embed_scenarios.py). Open this notebook from inside the Git folder.")

REPO = find_repo()
sys.path.insert(0, os.path.join(REPO, "src"))
from embed_scenarios import build_with_scenarios

out = OUT_FILE or os.path.join(REPO, "ewr_dashboard_with_scenarios.html")
html = build_with_scenarios(SCENARIOS, out, template=os.path.join(REPO, "index.html"))

In [ ]:
# The dashboard, shown in the notebook. Use the saved file for the full-size page (double-click it; it needs no server).
try:
    displayHTML(html)
except NameError:
    print("Open the saved file in a browser.")